In [1]:
import os
import pickle
import anndata
import numpy as np
import pandas as pd
import torch
from transformers import AutoConfig, AutoModel, AutoTokenizer

os.environ["HF_HOME"] = "/data/rbg/users/ashyamal/huggingface_cache"

In [2]:
DATASET = "CALGB"

RNA_counts_path = f"../../data/{DATASET}/RNA/{DATASET}_bulk_RNA_processed_log1p.h5ad"
bulkbertgenes_path = "../../data/BulkRNABert/common_gene_id.txt"
genecode_path = "../../data/GENCODE/gene_length.csv"

print(RNA_counts_path)
print(bulkbertgenes_path)
print(genecode_path)

../../data/CALGB/RNA/CALGB_bulk_RNA_processed_log1p.h5ad
../../data/BulkRNABert/common_gene_id.txt
../../data/GENCODE/gene_length.csv


In [3]:
genes = [g.replace("\n", "").strip() for g in open(bulkbertgenes_path)]

RNAseq_info = anndata.read_h5ad(RNA_counts_path)
gencode = pd.read_csv(genecode_path)

gene_symbol = gencode["gene_name"].values.tolist()
gene_id = gencode["gene_id"].values.tolist()
mapping = {gene_symbol[i]: gene_id[i].split(".")[0] for i in range(len(gene_symbol))}

RNA_counts = RNAseq_info.to_df()
RNA_counts.columns = [mapping.get(c, "-") for c in RNA_counts.columns]
print("Before dropping unmapped genes:", RNA_counts.shape)

RNA_counts = RNA_counts.loc[:, [c != "-" for c in RNA_counts.columns]]
print("After dropping unmapped genes:", RNA_counts.shape)

RNA = RNA_counts.reindex(columns=list(genes), fill_value=0.0)
print("After reindexing to BulkRNABert genes:", RNA.shape)

obs = "patient_id"
ids = RNAseq_info.obs.loc[:, obs].astype(str).tolist()

print(f"Number of CALGB samples: {len(ids)}")
print("First 5 IDs:", ids[:5])

Before dropping unmapped genes: (389, 34497)
After dropping unmapped genes: (389, 25087)
After reindexing to BulkRNABert genes: (389, 19062)
Number of CALGB samples: 389
First 5 IDs: ['SRR10488183', 'SRR10488017', 'SRR10488060', 'SRR10488167', 'SRR10488110']


In [4]:
gene_expression_array = RNA.to_numpy()
gene_expression_array = gene_expression_array / np.log(10)  # ln(TPM+1) -> log10(TPM+1)

print("Full array shape:", gene_expression_array.shape)
print("Matched genes:", RNA_counts.columns.isin(genes).sum(), "/", len(genes))
print("Zero-filled genes:", (gene_expression_array == 0).all(axis=0).sum(), "/", gene_expression_array.shape[1])

x0 = gene_expression_array[0:1]   # first sample only
id0 = ids[0]

print("First sample ID:", id0)
print("First sample input shape:", x0.shape)
print("Nonzero fraction:", (x0 > 0).mean())
print("Min / max:", x0.min(), x0.max())

Full array shape: (389, 19062)
Matched genes: 16736 / 19062
Zero-filled genes: 2326 / 19062
First sample ID: SRR10488183
First sample input shape: (1, 19062)
Nonzero fraction: 0.8606651977756794
Min / max: 0.0 4.219175099713102


In [5]:
device = torch.device("cpu")  # keep same as your script for exact reproducibility
print("device:", device)

config = AutoConfig.from_pretrained(
    "InstaDeepAI/BulkRNABert",
    trust_remote_code=True,
)
config.embeddings_layers_to_save = (4,)

tokenizer = AutoTokenizer.from_pretrained(
    "InstaDeepAI/BulkRNABert",
    trust_remote_code=True,
)

model = AutoModel.from_pretrained(
    "InstaDeepAI/BulkRNABert",
    config=config,
    trust_remote_code=True,
).to(device)

model.eval();

device: cpu


config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

bulkrnabert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/BulkRNABert:
- bulkrnabert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

tokenizer.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/BulkRNABert:
- tokenizer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


special_tokens_map.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/24.0M [00:00<?, ?B/s]

In [6]:
gene_expression_ids_0 = tokenizer.batch_encode_plus(
    x0,
    return_tensors="pt"
)["input_ids"]

print("Tokenized input_ids shape:", gene_expression_ids_0.shape)
print("config.n_genes:", config.n_genes)
assert x0.shape[1] == config.n_genes

Tokenized input_ids shape: torch.Size([1, 19062])
config.n_genes: 19062


In [7]:
with torch.no_grad():
    out = model(gene_expression_ids_0.to(device))

print(type(out))
print(out.keys())
for k, v in out.items():
    if hasattr(v, "shape"):
        print(k, tuple(v.shape))
    else:
        print(k, type(v))

<class 'dict'>
dict_keys(['embeddings_4', 'logits'])
embeddings_4 (1, 19062, 256)
logits (1, 19062, 66)


In [9]:
with torch.no_grad():
    emb4 = model(gene_expression_ids_0.to(device))["embeddings_4"]
    pooled = emb4.mean(axis=1)

print("embeddings_4 shape:", tuple(emb4.shape))
print("pooled shape:", tuple(pooled.shape))

embeddings_4 shape: (1, 19062, 256)
pooled shape: (1, 256)


In [10]:
script_style_emb = pooled.detach().cpu().numpy()
print(script_style_emb.shape)
script_style_emb[:1, :5]

(1, 256)


array([[-0.63513404,  0.276041  , -0.14307074,  0.08886844,  0.09385284]],
      dtype=float32)

In [11]:
gene_level_emb = emb4.squeeze(0).detach().cpu().numpy()  # shape [G, D]

print("gene-level matrix shape:", gene_level_emb.shape)
print("number of BulkRNABert genes:", len(genes))
print("Does G match len(genes)?", gene_level_emb.shape[0] == len(genes))
print("Hidden dim:", gene_level_emb.shape[1])

gene-level matrix shape: (19062, 256)
number of BulkRNABert genes: 19062
Does G match len(genes)? True
Hidden dim: 256


In [12]:
recomputed_pooled = gene_level_emb.mean(axis=0)
direct_pooled = script_style_emb.squeeze(0)

print("recomputed pooled shape:", recomputed_pooled.shape)
print("direct pooled shape:", direct_pooled.shape)
print("max abs diff:", np.max(np.abs(recomputed_pooled - direct_pooled)))
print("allclose:", np.allclose(recomputed_pooled, direct_pooled, atol=1e-6))

recomputed pooled shape: (256,)
direct pooled shape: (256,)
max abs diff: 4.196167e-05
allclose: True


In [13]:
RNAseq_info

AnnData object with n_obs × n_vars = 389 × 34497
    obs: 'patient_id'
    var: 'gene_symbol', 'log_variance'

In [17]:
# Compare multiple HVG pooling cutoffs for the same sample

top_ks = [1250, 5000, 10000]
results = []

gene_to_idx = {g: i for i, g in enumerate(genes)}

# Build mapped HVG dataframe once
orig_gene_symbols = RNAseq_info.var_names.astype(str).tolist()
mapped_gene_ids = [mapping.get(g, "-") for g in orig_gene_symbols]
lv = RNAseq_info.var["log_variance"].to_numpy()

hvg_df = pd.DataFrame({
    "orig_gene_symbol": orig_gene_symbols,
    "mapped_gene_id": mapped_gene_ids,
    "log_variance": lv,
})

hvg_df = hvg_df[hvg_df["mapped_gene_id"] != "-"].copy()
hvg_df = (
    hvg_df.sort_values("log_variance", ascending=False)
          .drop_duplicates(subset="mapped_gene_id", keep="first")
)
hvg_df = hvg_df[hvg_df["mapped_gene_id"].isin(gene_to_idx)].copy()

full_pooled = gene_level_emb.mean(axis=0)

for top_k in top_ks:
    top_hvg_ids = hvg_df.nlargest(top_k, "log_variance")["mapped_gene_id"].tolist()
    hvg_indices = [gene_to_idx[g] for g in top_hvg_ids]
    hvg_pooled = gene_level_emb[hvg_indices].mean(axis=0)

    cosine = np.dot(full_pooled, hvg_pooled) / (
        np.linalg.norm(full_pooled) * np.linalg.norm(hvg_pooled)
    )
    l2 = np.linalg.norm(full_pooled - hvg_pooled)

    results.append({
        "top_k": top_k,
        "n_found": len(hvg_indices),
        "cosine_to_full": float(cosine),
        "l2_to_full": float(l2),
    })

pd.DataFrame(results)

,top_k,n_found,cosine_to_full,l2_to_full
0,1250,1250,0.978086,6.176336
1,5000,5000,0.985124,5.019261
2,10000,10000,0.987310,4.450418
